# Merging new SOC results
This notebook is to merge all new results generated using the new FAO GAEZ yields as inputs, including reduced tillage and just crops.

## SETUP

### Modules

In [51]:
import pandas as pd
import geopandas as gpd
import sbtn_leaf.map_plotting as mp
import sbtn_leaf.map_calculations as mc
import sbtn_leaf.paths as sbtn_path
import sqlite3
import fiona

LEAFS_FOLDER = sbtn_path._LEAFS_DIR
PROJECT_ROOT = sbtn_path.project_root()

### Data

Data paths

In [52]:
# GEOPACKAGES
gpckg_crop_ct_path   = LEAFS_FOLDER / "SOC/SOC_2030_country_crops_clipped.gpkg"
gpckg_crop_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_subcountry_crops_clipped.gpkg"
gpckg_crop_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_ecoregions_crops_clipped.gpkg"

gpckg_fg_ct_path   = LEAFS_FOLDER / "SOC/SOC_2030_country_forest_grass.gpkg"
gpckg_fg_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_subcountry_forest_grass.gpkg"
gpckg_fg_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_ecoregions_forest_grass.gpkg"

gpckg_rt_ct_path   = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_country.gpkg"
gpckg_rt_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_subcountry.gpkg"
gpckg_rt_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_ecoregions.gpkg"

# Geopackages layers names
ct_layer = "soc_leaf_country"
sc_layer = "soc_leaf_subcountry"
er_layer = "soc_leaf_ecoregions"
geom_layer = "geometry_layer"

# csvs
csv_crop_ct_path   = LEAFS_FOLDER / "SOC/SOC_2030_country_crops_clipped.csv"
csv_crop_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_subcountry_crops_clipped.csv"
csv_crop_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_ecoregions_crops_clipped.csv"

csv_fg_ct_path   = LEAFS_FOLDER / "SOC/SOC_2030_country_forest_grass.csv"
csv_fg_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_subcountry_forest_grass.csv"
csv_fg_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_ecoregions_forest_grass.csv"

csv_rt_ct_path   = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_country.csv"
csv_rt_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_subcountry.csv"
csv_rt_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_ecoregions.csv"

Original geometries

In [53]:
country_shp = gpd.read_file(PROJECT_ROOT/'data/CountryLayers/Country_Level0/g2015_2014_0_dissolved.shp')
sc_shp = gpd.read_file(PROJECT_ROOT/'data/CountryLayers/SubCountry_Level1/g2015_2014_1.shp')
er_shp = gpd.read_file(PROJECT_ROOT/'data/Ecoregions2017/Ecoregions2017.shp')

### Opening geopackages

Countries

In [74]:
crop_country_geom =   gpd.read_file(gpckg_crop_ct_path, layer=geom_layer)
crop_country_values = gpd.read_file(gpckg_crop_ct_path, layer=ct_layer)
crop_country_values = crop_country_values.drop(columns = "_source_file")

In [55]:
fg_country_geom =  gpd.read_file(gpckg_fg_ct_path, layer=geom_layer)
fg_country_values = gpd.read_file(gpckg_fg_ct_path, layer=ct_layer)
fg_country_values = fg_country_values.drop(columns = "_source_file")

In [47]:
rt_country_geom =  gpd.read_file(gpckg_rt_ct_path, layer=geom_layer)
rt_country_values = gpd.read_file(gpckg_rt_ct_path, layer=ct_layer)
rt_country_values = rt_country_values.drop(columns = "_source_file")

Subcountries

In [86]:
crop_subcountry_geom =   gpd.read_file(gpckg_crop_sc_path, layer=geom_layer)
crop_subcountry_values = gpd.read_file(gpckg_crop_sc_path, layer=sc_layer)
crop_subcountry_values = crop_subcountry_values.drop(columns = "_source_file")

In [87]:
fg_subcountry_geom =  gpd.read_file(gpckg_fg_sc_path, layer=geom_layer)
fg_subcountry_values = gpd.read_file(gpckg_fg_sc_path, layer=sc_layer)
fg_subcountry_values = fg_subcountry_values.drop(columns = "_source_file")

In [88]:
rt_subcountry_geom =  gpd.read_file(gpckg_rt_sc_path, layer=geom_layer)
rt_subcountry_values = gpd.read_file(gpckg_rt_sc_path, layer=sc_layer)
rt_subcountry_values = rt_subcountry_values.drop(columns = "_source_file")

Ecoregions

In [89]:
crop_er_geom =   gpd.read_file(gpckg_crop_er_path, layer=geom_layer)
crop_er_values = gpd.read_file(gpckg_crop_er_path, layer=er_layer)
crop_er_values = crop_er_values.drop(columns = "_source_file")

In [90]:
fg_er_geom =  gpd.read_file(gpckg_fg_er_path, layer=geom_layer)
fg_er_values = gpd.read_file(gpckg_fg_er_path, layer=er_layer)
fg_er_values = fg_er_values.drop(columns = "_source_file")

In [91]:
rt_er_geom =  gpd.read_file(gpckg_rt_er_path, layer=geom_layer)
rt_er_values = gpd.read_file(gpckg_rt_er_path, layer=er_layer)
rt_er_values = rt_er_values.drop(columns = "_source_file")

## Merging
As the geometries have been stored in a separate layer, we just need to update the values layer, and rename it

### Countries

Creating the new geopackage with the geom layer

In [75]:
output_path_gpkg = LEAFS_FOLDER/"SOC/SOC_2030_country_v1.0.gpkg"
output_path_csv = LEAFS_FOLDER/"SOC/SOC_2030_country_v1.0.csv"
country_values_layer = "soc_leafs_country"

In [76]:
crop_country_geom.to_file(output_path_gpkg, layer=geom_layer, driver="GPKG")

Stacking all values

In [77]:
country_values_stacked = pd.concat([crop_country_values,fg_country_values,rt_country_values], ignore_index=True)

Renaming cf_std to leaf_std etc.

In [78]:
country_values_stacked = country_values_stacked.rename(columns = {"cf": "leaf", "cf_median": "leaf_median", "cf_std": "leaf_std"})

In [79]:
country_values_stacked.columns

Index(['ADM0_NAME', 'flow_name', 'leaf', 'leaf_median', 'leaf_std'], dtype='object')

Writing the new values layer

In [80]:
# Write values table directly into the geopackage (which is just a sqlite db)
conn = sqlite3.connect(output_path_gpkg)
country_values_stacked.to_sql(country_values_layer, conn, if_exists="replace", index=False)
conn.close()

Checking all flows are there

In [81]:
country_values_stacked_long = country_values_stacked.melt(id_vars=["ADM0_NAME", "flow_name"])

In [82]:
country_values_stacked_long.to_csv(output_path_csv, header=True)

Putting this all together in a function

In [98]:
def write_gpckg(geometry_layer: gpd.GeoDataFrame, output_path_gpkg: str, output_path_csv: str, value_layer_name: str, values_df_list: list, id_geom_name: str, rename_columns= True, geom_layer_name = geom_layer):
    # Write the geometry layer
    geometry_layer.to_file(output_path_gpkg, layer = geom_layer_name, driver = "GPKG")

    # Stacking values
    values_stacked = pd.concat(values_df_list, ignore_index=True)

    # Renaming
    if rename_columns:
        values_stacked = values_stacked.rename(columns = {"cf": "leaf", "cf_median": "leaf_median", "cf_std": "leaf_std"})

    # Write values table directly into the geopackage (which is just a sqlite db)
    conn = sqlite3.connect(output_path_gpkg)
    values_stacked.to_sql(value_layer_name, conn, if_exists="replace", index=False)
    conn.close()

    # Write csv
    values_stacked_long = values_stacked.melt(id_vars=[id_geom_name, "flow_name"])
    values_stacked_long.to_csv(output_path_csv, header=True)

    print(f"Geopackage saved into {output_path_gpkg} and csv saved into {output_path_csv}")

    return values_stacked_long

### Subcountry

Updating to subcountry variables

In [95]:
output_path_gpkg = LEAFS_FOLDER/"SOC/SOC_2030_subcountry_v1.0.gpkg"
output_path_csv = LEAFS_FOLDER/"SOC/SOC_2030_subcountry_v1.0.csv"
values_layer_name = "soc_leafs_subcountry"
values_df_list = [crop_subcountry_values,fg_subcountry_values,rt_subcountry_values]
id_geom_name = "ADM1_CODE"
geometry_layer=crop_subcountry_geom

Running the function

In [100]:
subcountry_stacked_values = write_gpckg(
    geometry_layer=geometry_layer,
    output_path_gpkg=str(output_path_gpkg),
    output_path_csv = str(output_path_csv),
    value_layer_name=values_layer_name,
    values_df_list=values_df_list,
    id_geom_name=id_geom_name
)

Geopackage saved into C:\Users\loyola\OneDrive - World Wildlife Fund, Inc\Documents\203. Python projects\SBTN_Test\LEAFs\SOC\SOC_2030_subcountry_v1.0.gpkg and csv saved into C:\Users\loyola\OneDrive - World Wildlife Fund, Inc\Documents\203. Python projects\SBTN_Test\LEAFs\SOC\SOC_2030_subcountry_v1.0.csv


In [101]:
subcountry_stacked_values

,ADM1_CODE,flow_name,variable,value
0,40542,Apple_irr_2030y_SOC,leaf,NaN
1,40543,Apple_irr_2030y_SOC,leaf,NaN
2,40544,Apple_irr_2030y_SOC,leaf,NaN
3,40545,Apple_irr_2030y_SOC,leaf,NaN
4,40546,Apple_irr_2030y_SOC,leaf,NaN
...,...,...,...,...
1129255,1273,Wheat_rf_ron_rt_2030y_SOC,leaf_std,NaN
1129256,2369,Wheat_rf_ron_rt_2030y_SOC,leaf_std,NaN
1129257,2241,Wheat_rf_ron_rt_2030y_SOC,leaf_std,NaN
1129258,980,Wheat_rf_ron_rt_2030y_SOC,leaf_std,NaN


Now need to update to actually get the needed data (country, subcountry) into the csv

In [102]:
sc_shp.columns

Index(['ADM1_CODE', 'ADM1_NAME', 'STR1_YEAR', 'EXP1_YEAR', 'STATUS',
       'DISP_AREA', 'ADM0_CODE', 'ADM0_NAME', 'SHAPE_LENG', 'SHAPE_AREA',
       'geometry'],
      dtype='object')

In [106]:
sc_df = subcountry_stacked_values.merge(sc_shp[["ADM0_NAME", "ADM1_NAME", "ADM1_CODE"]], how = "left", on = "ADM1_CODE")

In [110]:
sc_df = sc_df[["ADM0_NAME", "ADM1_NAME", "ADM1_CODE","flow_name","variable","value"]]

In [112]:
sc_df.to_csv(output_path_csv, header=True)

### Ecoregions

Updating  variables

In [120]:
output_path_gpkg = LEAFS_FOLDER/"SOC/SOC_2030_ecoregions_v1.0.gpkg"
output_path_csv = LEAFS_FOLDER/"SOC/SOC_2030_ecoregions_v1.0.csv"
values_layer_name = "soc_leafs_ecoregions"
values_df_list = [crop_er_values,fg_er_values,rt_er_values]
id_geom_name = "ECO_ID"
geometry_layer=crop_er_geom

Running the function

In [121]:
er_stacked_values = write_gpckg(
    geometry_layer=geometry_layer,
    output_path_gpkg=str(output_path_gpkg),
    output_path_csv = str(output_path_csv),
    value_layer_name=values_layer_name,
    values_df_list=values_df_list,
    id_geom_name=id_geom_name
)

Geopackage saved into C:\Users\loyola\OneDrive - World Wildlife Fund, Inc\Documents\203. Python projects\SBTN_Test\LEAFs\SOC\SOC_2030_ecoregions_v1.0.gpkg and csv saved into C:\Users\loyola\OneDrive - World Wildlife Fund, Inc\Documents\203. Python projects\SBTN_Test\LEAFs\SOC\SOC_2030_ecoregions_v1.0.csv


Need to delete values for Antarctica, as this is an artificial value based on the Tundra Biome, which is also filtered out

In [148]:
er_geom2 = gpd.read_file(output_path_gpkg, layer = geom_layer)
er_values2 = gpd.read_file(output_path_gpkg, layer = values_layer_name)

In [149]:
er_geom2_filtered = er_geom2[(er_geom2["REALM"]!="Antarctica") & (er_geom2["BIOME_NAME"]!="Tundra")]

Now deleting the values from the value df

In [ ]:
TundraIDs = er_geom2[er_geom2["BIOME_NAME"]=="Tundra"][["ECO_ID"]].drop_duplicates().sort_values(by="ECO_ID")

In [157]:
TundraIDs

,ECO_ID
0,117
148,118
840,119
210,120
245,121
246,122
248,123
841,124
488,125
480,126


In [159]:
er_values2_filtered = er_values2[~er_values2["ECO_ID"].isin(TundraIDs["ECO_ID"])]

And now actually I can just run the function again

In [163]:
output_path_gpkg = LEAFS_FOLDER/"SOC/SOC_2030_ecoregions_v1.0.gpkg"
output_path_csv = LEAFS_FOLDER/"SOC/SOC_2030_ecoregions_v1.0.csv"
values_layer_name = "soc_leafs_ecoregions"
values_df_list = [er_values2_filtered]
id_geom_name = "ECO_ID"
geometry_layer=er_geom2_filtered

In [164]:
er_stacked_values = write_gpckg(
    geometry_layer=geometry_layer,
    output_path_gpkg=str(output_path_gpkg),
    output_path_csv = str(output_path_csv),
    value_layer_name=values_layer_name,
    values_df_list=values_df_list,
    id_geom_name=id_geom_name
)

Geopackage saved into C:\Users\loyola\OneDrive - World Wildlife Fund, Inc\Documents\203. Python projects\SBTN_Test\LEAFs\SOC\SOC_2030_ecoregions_v1.0.gpkg and csv saved into C:\Users\loyola\OneDrive - World Wildlife Fund, Inc\Documents\203. Python projects\SBTN_Test\LEAFs\SOC\SOC_2030_ecoregions_v1.0.csv
